# Discrete Rigid Contact


## Goal
Compare co-simulation with a monolithic Modelica reference for rigid wall contact, and show convergence as communication step size decreases.

## Model Map
- `SineAngleSetPoint` → `PIDContinuous` → `DriveDynamic` → `FMUPendulum`
- Measurement path: `FMUPendulum.theta` → `AnglePotentiometerADC` → `AnglePotentiometerADCDecoder` → `PIDContinuous`
- Discrete contact: event indicator at wall angle, inverts velocity

## Assumptions and Scope
- Rigid wall contact (event-driven)
- Fixed-step co-simulation; reference uses variable-step DASSL

## Prerequisites and Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

from OMPython import ModelicaSystem
from demos.ControlledPendulum.src.master_pendulum import MasterPendulum, FMUPendulum
import demos.ControlledPendulum.src.master_pendulum.components.fem.pendulum_config as config

from syssimx import FMUComponent, System, SystemGraphVisualizer, Connection, EventConnection

## Discover FMUs

In [ ]:
PLATFORM = sys.platform
demo_dir_path = Path().cwd() / "demos" / "ControlledPendulum"
package_path = Path(demo_dir_path / "src/modelica/ControlledPendulum")
fmu_output_dir = Path(demo_dir_path / f"artifacts/fmus/{PLATFORM}")

fmu_paths = {}
for subdir in fmu_output_dir.iterdir():
    if subdir.is_dir():
        fmu_paths[subdir.name] = {}
        for fmu_file in subdir.glob("*.fmu"):
            fmu_paths[subdir.name][fmu_file.stem] = fmu_file
    else:
        fmu_paths[subdir.stem] = subdir


## Modelica Reference Simulation (Monolithic)

In [ ]:
pkg_path = demo_dir_path / 'src/modelica/ControlledPendulum/package.mo'
model_name = 'ControlledPendulum.Examples.Contact.RigidContact'
ref_system = ModelicaSystem(str(pkg_path), model_name)

ref_system.setParameters("useReset=false")
ref_system.buildModel()
ref_system.simulate()
ref_names = ('time', 'theta', 'theta_ref', 'theta_meas', 'pid.I_out')
sol_no_reset = {name: ref_system.getSolutions(name).flatten() for name in ref_names}

ref_system.setParameters("useReset=true")
ref_system.buildModel()
ref_system.simulate()
ref_names = ('time', 'theta', 'theta_ref', 'theta_meas', 'pid.I_out')
sol_with_reset = {name: ref_system.getSolutions(name).flatten() for name in ref_names}


## Instantiate FMU Components

In [ ]:
set_point = FMUComponent(name="SineAngle",
                         fmu_path=fmu_paths['Trajectories']["SineAngleSetPoint"],
                         group="Set Point")

drive = FMUComponent(name="Drive",
                     fmu_path=fmu_paths['Actuators']["DriveDynamic"],
                     group="Actuator")

angle_encoder = FMUComponent(name="Angle Sensor",
                             fmu_path=fmu_paths['Sensors']['AnglePotentiometerADC'],
                             group="Encoder")

angle_decoder = FMUComponent(name="Decoder",
                             fmu_path=fmu_paths['Sensors']['AnglePotentiometerADCDecoder'],
                             group="Decoder")

class PIDController(FMUComponent):
    def __init__(self, name):
        fmu_path = fmu_paths['Controllers']['PIDContinuous_euler']
        super().__init__(name=name, fmu_path=fmu_path, group="Controller")
        self.use_reset = False  # Flag to control whether to apply reset on event
    
    def _handle_events_internal(self, event_names, t):
        if not self.use_reset:
            return  # Skip handling events if reset is disabled
        if "wall_hit" not in event_names:
            return
        self.set_inputs({"resetI": True})
        self.do_step(t, 0)  # Perform a step to apply the resetI input
        self.set_inputs({"resetI": False})

pid = PIDController(name="PID")

pendulum = FMUPendulum(name="Pendulum",solver="cvode")

def wall_contact_indicator(comp: FMUPendulum) -> float:
    theta = comp.get_outputs()["theta"]
    theta_wall = 0
    return theta - theta_wall

pendulum.add_event_indicator("wall_hit", func=wall_contact_indicator, direction=-1)


## Define Connections

In [ ]:
c1 = Connection(
    src_comp=set_point.name,
    src_port=set_point.output_specs["theta_ref"].name,
    dst_comp=pid.name,
    dst_port=pid.input_specs["theta_ref"].name,
)

c21 = Connection(
    src_comp=pendulum.name,
    src_port=pendulum.output_specs["theta"].name,
    dst_comp=angle_encoder.name,
    dst_port=angle_encoder.input_specs["theta"].name,
)

c22 = Connection(
    src_comp=angle_encoder.name,
    src_port=angle_encoder.output_specs["v_out"].name,
    dst_comp=angle_decoder.name,
    dst_port=angle_decoder.input_specs["v_in"].name,
)

c23 = Connection(
    src_comp=angle_decoder.name,
    src_port=angle_decoder.output_specs["theta"].name,
    dst_comp=pid.name,
    dst_port=pid.input_specs["theta_meas"].name,
)

c3 = Connection(
    src_comp=pid.name,
    src_port=pid.output_specs["u"].name,
    dst_comp=drive.name,
    dst_port=drive.input_specs["u_control"].name,
)

c4 = Connection(
    src_comp=drive.name,
    src_port=drive.output_specs["torque"].name,
    dst_comp=pendulum.name,
    dst_port=pendulum.input_specs["tau"].name,
)

c5 = Connection(
    src_comp=pendulum.name,
    src_port=pendulum.output_specs["omega"].name,
    dst_comp=drive.name,
    dst_port=drive.input_specs["omega"].name,
)

event_connection_1 = EventConnection(
    src_comp=pendulum.name,
    src_port="wall_hit",
    dst_comp=pendulum.name,
    dst_port=pendulum.input_specs["omega_invert"].name,
)

event_connection_2 = EventConnection(
    src_comp=pendulum.name,
    src_port="wall_hit",
    dst_comp=pid.name,
    dst_port=pid.input_specs["resetI"].name,
)

connections = [c1, c21, c22, c23, c3, c4, c5]
components  = [set_point, pid, drive, pendulum, angle_decoder, angle_encoder]

## Build and Initialize the System - No Reset

In [ ]:
system = System(name="Controlled Pendulum System")

for comp in components:
    system.add_component(comp)

for conn in connections:
    system.add_connection(conn)

system.add_event_connection(event_connection_1)
system.add_event_connection(event_connection_2)

system.initialize(t0=0.0)


## Visualize the System Graph

In [ ]:
visualizer = SystemGraphVisualizer(system)
visualizer.visualize()

In [ ]:
t = 0
t_end = 5
dt = 0.01

system.reset()
system.initialize(t0=t)
system.run(t0=t, tf=t_end, dt=dt)

In [ ]:
history = system.get_history()

t_set_point = history[set_point.name][0]
theta_ref = history[set_point.name][1]['theta_ref']

t_pendulum = history[pendulum.name][0]
theta_pendulum = history[pendulum.name][1]['theta']

t_pid = history[pid.name][0]
I_out = history[pid.name][1]['I_out']

In [ ]:
plt.figure(figsize=(10, 6 ))
plt.subplot(2, 1, 1)
plt.plot(t_set_point, theta_ref, label='Set Point (Reference)', linestyle='--', color='red')
plt.plot(t_pendulum, theta_pendulum, label='Pendulum Angle (rad)')
# plt.plot(sol_no_reset['time'], sol_no_reset['theta'], label='Reference (No Reset)', linestyle='--')
plt.title('Pendulum Angle Over Time')
plt.xlabel('Time (s)')
plt.ylabel('Angle (rad)')
plt.grid()
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t_pid, I_out, label='PID Integral Term')
# plt.plot(sol_no_reset['time'], sol_no_reset['pid.I_out'], label='Reference Error (No Reset)', linestyle='--')
plt.title('PID Integral Term Over Time')
plt.xlabel('Time (s)')
plt.ylabel('Integral Term Value')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

## Summary
- Co-simulation trajectories converge toward the monolithic reference as `dt` decreases.
- Resetting the PID integrator reduces windup after contact events.